In [19]:
import numpy as np
import pandas as pd
import rasterio as rio
from matplotlib import pyplot as plt
from pathlib import Path

In [3]:
country_pseudo_values = pd.read_csv('../data/country-pseudo-values.csv')

In [16]:
country_raster_1km_ = rio.open(r"..\results\country-rasters\country-raster-eck4-1km.tif")
country_raster_1km = country_raster_1km_.read(1)

In [5]:
df_income_groups = pd.read_excel('../data/income-groups.xlsx', index_col=0)

In [11]:
group_values = dict({'L': 1, 'LM': 2, 'UM': 3, 'H': 4})

In [7]:
country_pseudo_values

,value,country,full name,region,GRIP_region,notes
0,1,IRN,Iran,Middle East,5.0,NaN
1,2,BLM,Saint-Barth??lemy,Latin America and the Caribbean,2.0,NaN
2,3,UGA,Uganda,Africa,3.0,NaN
3,4,SLE,Sierra Leone,Africa,3.0,NaN
4,5,VCT,Saint Vincent and the Grenadines,Latin America and the Caribbean,2.0,NaN
...,...,...,...,...,...,...
250,251,MNP,Northern Mariana Islands,Oceania,7.0,NaN
251,252,IOT,British Indian Ocean Territory,Africa,3.0,NaN
252,253,ARM,Armenia,Central and Northern Asia,5.0,Western Asia
253,254,HKG,Hong Kong,Eastern Asia,6.0,NaN


In [18]:
country_pseudo_values[country_pseudo_values['country'] == 'CHI']

,value,country,full name,region,GRIP_region,notes


In [12]:
income_group_raster = np.zeros(country_raster_1km_.shape, dtype=np.uint8)

In [17]:
for country in df_income_groups.index:
    try:
        the_value = country_pseudo_values[country_pseudo_values['country'] == country]['value'].values[0]
        the_group = df_income_groups.loc[country, 'group']
        the_group_value = group_values[the_group]
        # print(country, the_value, the_group)
        the_country_mask = country_raster_1km == the_value
        income_group_raster[the_country_mask] = the_group_value
    except:
        print(country)
    # if country not in country_pseudo_values.index:
    #     print(country)

CHI
XKX


In [20]:
Path(r'../results/income-group-rasters').mkdir(parents=True, exist_ok=True)

In [21]:
with rio.open(
    r'../results/income-group-rasters/income-group-raster-eck4-1km.tif',
    'w',
    driver='GTiff',
    height=country_raster_1km_.shape[0],
    width=country_raster_1km_.shape[1],
    count=1,
    dtype=income_group_raster.dtype, 
    # crs=raster_1km_.crs,
    crs=country_raster_1km_.crs,
    transform=country_raster_1km_.transform,
    compress='lzw',
) as dst:
    dst.write(income_group_raster, 1)